# **Train YOLOv12-Pose on AwA2 dataset**

## ***Create YOLO dataset***

In [ ]:
!invidia-smi

In [ ]:
!pip install ultralytics onnx onnxruntime

In [ ]:
print(100*'#')
!pip show ultralytics
print(100*'-')
!pip show onnx
print(100*'-')
!pip show onnxruntime
print(100*'#')

### ***Imports***

In [ ]:
import os
import yaml
import base64
import io
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

### ***Konfiguration***

In [ ]:
DATASET_ROOT = os.path.join('/kaggle','input','awa2-dataset','AwA2_pose','lite_dataset')  # folder with three .parquet
OUTPUT_ROOT  = os.path.join('/kaggle','working','awa2_yolo_pose')    # where to save the final dataset
IMG_SUBFOLDER = "images"
LBL_SUBFOLDER = "labels"

In [ ]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_ROOT,IMG_SUBFOLDER), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_ROOT,LBL_SUBFOLDER), exist_ok=True)

#### ***Creating folders***

In [ ]:
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_ROOT,IMG_SUBFOLDER,split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT,LBL_SUBFOLDER,split), exist_ok=True)

#### ***Columns with keys***

In [ ]:
keypoint_columns = [
    'nose','belly_bottom',
    'right_eye','left_eye',
    'right_antler_base','left_antler_base',
    'right_antler_end','left_antler_end',
    'right_earend','left_earend',
    'right_earbase','left_earbase',
    'mouth_end_right','mouth_end_left',
    'lower_jaw','upper_jaw',
    'body_middle_right','body_middle_left',
    'front_right_knee','front_left_knee',
    'back_right_knee','back_left_knee',
    'front_right_paw','front_left_paw',
    'back_right_paw','back_left_paw',
    'front_right_thai','front_left_thai',
    'back_right_thai','back_left_thai',
    'throat_base','throat_end',
    'tail_base','tail_end',
    'neck_base','neck_end',
    'back_base','back_middle','back_end',
]

In [ ]:
NUM_KPTS = len(keypoint_columns)                  # z.B. 29–39
print(f"{NUM_KPTS} keypoints used")

#### ***Data retrieval***

In [ ]:
splits = {
    "train": pd.read_parquet(os.path.join(DATASET_ROOT,"train_lite.parquet")),
    "val":   pd.read_parquet(os.path.join(DATASET_ROOT,"validation_lite.parquet")),
    "test":  pd.read_parquet(os.path.join(DATASET_ROOT,"test_lite.parquet")),
}

####  ***Label encoder for classes (0–33)***

In [ ]:
all_classes = pd.concat([df["name_class"] for df in splits.values()]).unique()
le = LabelEncoder().fit(all_classes)

In [ ]:
print("Number of classes:", len(all_classes))
print("Example mapping:", dict(zip(all_classes[:5], le.transform(all_classes[:5]))))

### ***Main conversion***

In [ ]:
for split_name, df in splits.items():
    print(f"\nProcessing split: {split_name} ({len(df)} images)")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # 1. Decode the image
        try:
            img_bytes = base64.b64decode(row["image_base64s"])
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception as e:
            print(f"Error decoding image {row.get('name_file', idx)}: {e}")
            continue

        w, h = row["image_width"], row["image_height"]
        assert img.size == (w, h), "The dimensions don't fit!"

        # Save image
        img_name = f"{row.get('name_file', f'img_{idx:06d}')}.jpg"
        img_path = os.path.join(OUTPUT_ROOT,IMG_SUBFOLDER,split_name,img_name)
        img.save(img_path)

        # 2. Bounding box – [x1,y1,x2,y2] → YOLO normalized format
        if "bbox" not in row or len(row["bbox"]) != 4:
            continue  # skip without bbox
        x1, y1, x2, y2 = row["bbox"]
        xc = (x1 + x2) / 2 / w
        yc = (y1 + y2) / 2 / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        # 3. Key points + visibility
        kpts_norm = []
        for kp_col in keypoint_columns:
            kp_value = row.get(kp_col, [-1, -1])
        
            if kp_value is None or (isinstance(kp_value, float) and np.isnan(kp_value)):
                kp = [-1, -1]
            else:
                kp = kp_value
        
            # Format check
            if not isinstance(kp, (list, tuple, np.ndarray)) or len(kp) != 2:
                kpts_norm.extend([0.0, 0.0, 0])
                continue
        
            # Convert to a regular sheet (just in case)
            kp = list(kp)
        
            x, y = kp
        
            if x < 0 or y < 0:
                x_norm = 0.0
                y_norm = 0.0
                vis = 0
            else:
                x_norm = float(np.clip(x / w, 0.0, 1.0))
                y_norm = float(np.clip(y / h, 0.0, 1.0))
                vis = 2
        
            kpts_norm.extend([x_norm, y_norm, vis])

        # 4. class id
        class_id = le.transform([row["name_class"]])[0]

        #5. .txt writing
        label_path = os.path.join(OUTPUT_ROOT,LBL_SUBFOLDER,split_name, img_name.replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            line = f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
            for val in kpts_norm:
                if isinstance(val, float):
                    line += f" {val:.6f}"
                else:
                    line += f" {val}"
            f.write(line + "\n")

print("\nDone! Dataset ready in:", OUTPUT_ROOT)

### ***We will create the file awa2-pose-39kpt.yaml***

In [ ]:
df = pd.read_parquet(os.path.join(DATASET_ROOT,'train_lite.parquet'))
class_names = sorted(df["name_class"].unique().tolist())
print(len(class_names))          # should be 34
print(class_names)

### ***YAML file content***

In [ ]:
dataset_config = {
    "path": OUTPUT_ROOT,          # real absolute or relative path
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",       # optional
    
    "nc": 34,
    "names": class_names,
    
    "kpt_shape": [39, 3],
    
    "flip_idx": [3, 2, 5, 4, 7, 6, 9, 8, 11, 10, 13, 12,
                 17, 16, 19, 18, 21, 20, 23, 22, 25, 24, 27, 26, 29, 28],
    
    # augmentation parameters (optional - good for animals)
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.2,
    "scale": 0.5,
    "shear": 3.0,
    "perspective": 0.0,
    "flipud": 0.4,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.2,
}

### ***Save to file***

In [ ]:
yaml_path = os.path.join('/kaggle','working','awa2-pose-39kpt.yaml')

In [ ]:
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(
        dataset_config,
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False
    )

print(f"File created: {yaml_path}")